In [2]:
#Cell 1 : Load model and generate validation scores
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve

# ── Load model payload ──────────────────────────────────────────
with open("../models/lgbm_baseline_ieee.pkl", "rb") as f:
    payload = pickle.load(f)

model        = payload["model"]
feature_cols = payload["feature_cols"]

# ── Load validation set ─────────────────────────────────────────
val    = pd.read_parquet("../data/processed/features_val.parquet")
X_val  = val[feature_cols]
y_val  = val["isFraud"]

# ── Generate scores ─────────────────────────────────────────────
y_scores = model.predict(X_val)

print(f"Validation set size : {len(y_val):,}")
print(f"Fraud cases         : {y_val.sum():,} ({y_val.mean()*100:.2f}%)")
print(f"Score range         : {y_scores.min():.4f} – {y_scores.max():.4f}")
print(f"Score mean          : {y_scores.mean():.4f}")
print(f"Score median        : {np.median(y_scores):.4f}")


Validation set size : 118,108
Fraud cases         : 4,064 (3.44%)
Score range         : 0.0002 – 0.9983
Score mean          : 0.1510
Score median        : 0.0647


In [4]:
# Cell 2 : Threshold Sweep — Find FPR < 2% Block Threshold + Safe Approve Threshold

thresholds = np.linspace(0.001, 0.999, 1000)
results = []

for t in thresholds:
    preds = (y_scores >= t).astype(int)
    tp = ((preds == 1) & (y_val == 1)).sum()
    fp = ((preds == 1) & (y_val == 0)).sum()
    tn = ((preds == 0) & (y_val == 0)).sum()
    fn = ((preds == 0) & (y_val == 1)).sum()

    fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    # Fraud contamination: what % of APPROVED transactions are actually fraud?
    approved_fraud_rate = fn / (tn + fn) if (tn + fn) > 0 else 0

    results.append({"threshold": t, "fpr": fpr, "recall": rec,
                    "precision": prec, "f1": f1,
                    "approved_fraud_rate": approved_fraud_rate})

sweep = pd.DataFrame(results)

# ── threshold_block: lowest threshold where FPR drops under 2% ─
threshold_block = sweep[sweep["fpr"] < 0.02].iloc[0]["threshold"]
block_row       = sweep[sweep["fpr"] < 0.02].iloc[0]

# ── threshold_approve: highest threshold where approved fraud 
#    contamination stays below 0.5% (< 1/7th of base rate 3.44%)
approve_candidates = sweep[sweep["approved_fraud_rate"] < 0.005]
threshold_approve  = approve_candidates.iloc[-1]["threshold"]
approve_row        = approve_candidates.iloc[-1]

print("=== OPERATING THRESHOLDS ===")
print(f"threshold_approve : {threshold_approve:.4f}")
print(f"threshold_block   : {threshold_block:.4f}")
print(f"Sanity check      : approve < block → {threshold_approve < threshold_block}")
print()
print("=== AT threshold_approve (APPROVE/FLAG boundary) ===")
print(f"  Fraud in approved txns : {approve_row['approved_fraud_rate']*100:.3f}%")
print(f"  Recall                 : {approve_row['recall']*100:.2f}%")
print()
print("=== AT threshold_block (FLAG/BLOCK boundary) ===")
print(f"  FPR       : {block_row['fpr']*100:.2f}%   (target: < 2%)")
print(f"  Recall    : {block_row['recall']*100:.2f}%")
print(f"  Precision : {block_row['precision']*100:.2f}%")
print(f"  F1        : {block_row['f1']:.4f}")


=== OPERATING THRESHOLDS ===
threshold_approve : 0.1888
threshold_block   : 0.6973
Sanity check      : approve < block → True

=== AT threshold_approve (APPROVE/FLAG boundary) ===
  Fraud in approved txns : 0.500%
  Recall                 : 89.03%

=== AT threshold_block (FLAG/BLOCK boundary) ===
  FPR       : 2.00%   (target: < 2%)
  Recall    : 55.83%
  Precision : 49.90%
  F1        : 0.5270


In [5]:
# Cell 3 : Verify & Save Thresholds

print(sweep[sweep["fpr"] < 0.02].iloc[0][["threshold", "fpr", "recall", "precision", "f1"]])

THRESHOLD_APPROVE = 0.1888
THRESHOLD_BLOCK   = 0.6973

print(f"\nFinal thresholds locked:")
print(f"  THRESHOLD_APPROVE = {THRESHOLD_APPROVE}")
print(f"  THRESHOLD_BLOCK   = {THRESHOLD_BLOCK}")


threshold    0.697302
fpr          0.019975
recall       0.558317
precision    0.499010
f1           0.527000
Name: 697, dtype: float64

Final thresholds locked:
  THRESHOLD_APPROVE = 0.1888
  THRESHOLD_BLOCK   = 0.6973
